# Matching-Specification Robustness: Singapore FY2023

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TAB_DIR = ROOT / "outputs" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)
SEED = 607
N_PERM = 9_999
N_BOOT = 5_000
df = pd.read_parquet(ROOT / "data/processed/orbis_singapore_fy2023_clean.parquet")


## Prepares the matching sample

In [2]:
# Listing status as text, from the cleaning script's 1/0 flag
df["listing_status"] = df["listed"].map({1: "Listed", 0: "Unlisted"})

# The cleaning pipeline preserves incorporation_year, including Orbis
# records that contain only a four-digit year.
df["incorporation_year"] = pd.to_numeric(df["incorporation_year"], errors="coerce")

# Actual age in FY2023
df["firm_age"] = 2023 - df["incorporation_year"]
df.loc[df["firm_age"] < 0, "firm_age"] = np.nan
df["age_band"] = pd.cut(df["firm_age"], bins=[-1, 4, 9, 19, float("inf")],
                        labels=["0–4", "5–9", "10–19", "20+"])

# Account type: C (consolidated) vs U (unconsolidated)
df["consolidated"] = df["consolidation_code"].astype("string").str.strip().str.upper().str[0]
df["consolidated"] = df["consolidated"].where(df["consolidated"].isin(["C", "U"]))

# Analysis sample: positive assets only (log_assets = ln(assets_usd), checked = 0 difference)
eda = df[df["listing_status"].notna() & (df["assets_usd"] > 0)].copy()

print("\nFirms by status:\n", eda["listing_status"].value_counts())
print("\nMissing matching variables:")
print(eda.groupby("listing_status")[["nace2", "firm_age", "age_band", "consolidated", "log_assets"]]
         .apply(lambda g: g.isna().sum()))
eda = eda.loc[~eda["is_listed_subsidiary"].fillna(False)].copy()
assert eda["bvd_id"].notna().all() and eda["bvd_id"].is_unique
assert np.isfinite(eda["log_assets"].astype(float)).all()
assert np.allclose(eda["log_assets"].astype(float), np.log(eda["assets_usd"].astype(float)))



Firms by status:
 listing_status
Unlisted    47949
Listed        496
Name: count, dtype: int64

Missing matching variables:
                nace2  firm_age  age_band  consolidated  log_assets
listing_status                                                     
Listed              0         0         0             0           0
Unlisted            0         0         0             0           0


For each matched set, the gap is:
\[
D_i=\ln(\text{listed assets})-\text{mean}\bigl(\ln(\text{private-control assets})\bigr)
\]
A positive gap means the listed firm has larger assets than its matched controls’ geometric mean.

The reported median ratio is:
\[
\exp\bigl(\text{median}(D_i)\bigr)
\]
A ratio of 8 means the median within-set comparison is approximately eight times as large. It does not mean the median listed firm’s assets divided by the median private firm’s assets equals 8.

### Reusable matching and inference
For set i, Dᵢ is the listed firm's log assets minus the mean log assets of its controls. exp(median(D)) compares the listed firm with its controls' geometric mean; it is not the ratio of the two groups' medians. Sets have equal weight.

The share listed larger uses all matched sets. The one-sided exact sign test excludes zero gaps. The one-sided permutation test randomly assigns the listed label within each set, recomputes the median gap, and uses the plus-one correction. Its interpretation requires within-set label exchangeability. The 95% percentile interval resamples whole matched sets, conditional on the fitted matching; it does not capture uncertainty from selecting matches. The same seeds and simulation counts are used for each specification.

In [3]:
KEYS = ["nace2", "age_band", "consolidated"]
ID = "bvd_id"
MAX_CONTROLS = 3

def run_matching(data, max_age_gap=None, seed=607, max_controls=3):
    """Round-robin nearest-age matching, up to MAX_CONTROLS, without replacement.
    max_age_gap: if set, controls must be within this many years of the listed firm."""
    if max_controls < 1:
        raise ValueError("max_controls must be positive")
    rng = np.random.default_rng(seed)
    base = data.dropna(subset=["log_assets", "firm_age"] + KEYS)
    L_all = base[base["listing_status"] == "Listed"]
    pool = base[base["listing_status"] == "Unlisted"]
    pool_groups = {k: g for k, g in pool.groupby(KEYS, observed=True)}
    sets, unmatched, n_sets = [], [], 0

    for key, L in L_all.groupby(KEYS, observed=True):
        L = L.sample(frac=1, random_state=int(rng.integers(1_000_000)))
        P = pool_groups.get(key)
        if P is None:
            unmatched += [{ID: i, "reason": "no private firm in same stratum"} for i in L[ID]]
            continue
        avail = P.assign(tiebreak=rng.random(len(P)))
        picks = {idx: [] for idx in L.index}
        for _ in range(max_controls):                       # round-robin
            for idx, row in L.iterrows():
                if avail.empty:
                    break
                gap = (avail["firm_age"] - row["firm_age"]).abs()
                ok = avail.assign(age_gap=gap)
                if max_age_gap is not None:
                    ok = ok[ok["age_gap"] <= max_age_gap]
                if ok.empty:
                    continue
                best = ok.sort_values(["age_gap", "tiebreak"]).index[0]
                picks[idx].append((best, gap[best]))
                avail = avail.drop(best)                    # no replacement
        for idx, row in L.iterrows():
            if not picks[idx]:
                reason = ("no unused private firm within age limit" if max_age_gap is not None
                          else "stratum's private firms all used")
                unmatched.append({ID: row[ID], "reason": reason})
                continue
            n_sets += 1
            sets.append({"set_id": n_sets, "role": "listed", ID: row[ID],
                         "firm_age": row["firm_age"], "age_gap": 0.0,
                         "log_assets": row["log_assets"], **{k: row[k] for k in KEYS}})
            for c_idx, c_gap in picks[idx]:
                c = P.loc[c_idx]
                sets.append({"set_id": n_sets, "role": "private", ID: c[ID],
                             "firm_age": c["firm_age"], "age_gap": c_gap,
                             "log_assets": c["log_assets"], **{k: c[k] for k in KEYS}})

    if not sets:
        raise ValueError("No matched sets for this specification")
    ml = pd.DataFrame(sets)
    ml["n_controls"] = ml["set_id"].map(ml[ml["role"] == "private"].groupby("set_id").size())
    return ml, pd.DataFrame(unmatched, columns=[ID, "reason"]), L_all, pool



The run_matching() function:
1. Separates eligible listed firms from private controls.
2. Groups firms by industry, age band, and account type.
3. Randomises the listed firms’ order using seed 607.
4. Selects available private firms closest in age.
5. Applies an age caliper when specified.
6. Records matched sets and reasons some listed firms remain unmatched.

In [4]:
def infer_matches(ml, seed=SEED):
    listed = ml.loc[ml["role"].eq("listed")].set_index("set_id")
    private = ml.loc[ml["role"].eq("private")]
    assert private[ID].is_unique, "Controls must not be reused"
    assert listed[ID].is_unique
    assert ml.groupby("set_id")[KEYS].nunique().le(1).all().all()
    D = (listed["log_assets"] - private.groupby("set_id")["log_assets"].mean()).to_numpy(dtype=float)
    assert len(D) > 0 and np.isfinite(D).all()
    nz = D[D != 0]
    sign_p = stats.binomtest(int((nz > 0).sum()), len(nz), alternative="greater").pvalue if len(nz) else np.nan
    rng = np.random.default_rng(seed)
    perm_D = np.empty((N_PERM, len(D)))
    col = 0
    for m, grp in ml.groupby("n_controls"):
        m = int(m)
        Y = grp.sort_values(["set_id", "role"]).groupby("set_id")["log_assets"].apply(np.array)
        Y = np.vstack(Y.to_numpy()).astype(float)
        j = rng.integers(0, m + 1, size=(N_PERM, len(Y)))
        yj = Y[np.arange(len(Y)), j]
        perm_D[:, col:col + len(Y)] = yj - (Y.sum(axis=1) - yj) / m
        col += len(Y)
    median = np.median(D)
    perm_p = (1 + (np.median(perm_D, axis=1) >= median).sum()) / (N_PERM + 1)
    idx = rng.integers(0, len(D), size=(N_BOOT, len(D)))
    lo, hi = np.percentile(np.median(D[idx], axis=1), [2.5, 97.5])
    return {"Matched Sets": len(D), "Share Listed Larger": (D > 0).mean(),
            "Median Log Gap": median, "Median Ratio": np.exp(median),
            "CI Low": np.exp(lo), "CI High": np.exp(hi),
            "Permutation p-value": perm_p, "Sign-test p-value": sign_p,
            "Nonzero Gaps": len(nz), "Zero Gaps": int((D == 0).sum())}


| Result | Meaning |
|---|---|
| Matched sets | Number of listed firms successfully matched |
| Share listed larger | Fraction of sets with a positive gap |
| Median ratio | Exponentiated median log gap |
| Sign-test p-value | Tests whether positive gaps occur more often than 50%, excluding zero gaps |
| Permutation p-value | Tests whether the observed median gap is unusually large under within-set label reassignment |
| 95% confidence interval | Bootstrap interval for the median ratio |

## robustness checks

### Run all five specifications
The USD 1 threshold is inclusive and uses assets_usd, already converted from thousand USD by the cleaning pipeline. It is applied before matching. Age-caliper rows reproduce the specifications in the main notebook and add the complete permutation/sign-test inference. Missing matching variables are excluded consistently for all rows.

In [5]:
specifications = [
    ("Up to 3 controls", eda, 3, None),
    ("1 control", eda, 1, None),
    ("Assets ≥ USD 1", eda.loc[eda["assets_usd"].ge(1)].copy(), 3, None),
    ("Age gap ≤ 15 years", eda, 3, 15),
    ("Age gap ≤ 10 years", eda, 3, 10),
]
results, samples, matches = [], [], {}
for name, sample, controls, caliper in specifications:
    ml, unmatched, listed_pool, private_pool = run_matching(
        sample, max_age_gap=caliper, seed=SEED, max_controls=controls)
    assert ml["n_controls"].between(1, controls).all()
    if caliper is not None:
        assert ml["age_gap"].le(caliper).all()
    matches[name] = ml
    results.append({"Matching Specification": name, **infer_matches(ml)})
    samples.append({"Matching Specification": name,
                    "Listed before matching": int(sample["listing_status"].eq("Listed").sum()),
                    "Unlisted before matching": int(sample["listing_status"].eq("Unlisted").sum()),
                    "Eligible listed": len(listed_pool), "Eligible unlisted": len(private_pool),
                    "Matched listed": ml["set_id"].nunique(), "Unmatched listed": len(unmatched),
                    "Matched controls": int(ml["role"].eq("private").sum()),
                    "Excluded below USD 1": len(eda) - len(sample)})
    assert ml["set_id"].nunique() + len(unmatched) == len(listed_pool)
results = pd.DataFrame(results).set_index("Matching Specification")
sample_sizes = pd.DataFrame(samples).set_index("Matching Specification")
display(sample_sizes)
results.to_csv(TAB_DIR / "robustness_matching_results.csv")
sample_sizes.to_csv(TAB_DIR / "robustness_matching_sample_sizes.csv")


,Listed before matching,Unlisted before matching,Eligible listed,Eligible unlisted,Matched listed,Unmatched listed,Matched controls,Excluded below USD 1
Matching Specification,,,,,,,,
Up to 3 controls,496,45952,496,45952,351,145,777,0
1 control,496,45952,496,45952,351,145,351,0
Assets ≥ USD 1,496,45748,496,45748,351,145,777,204
Age gap ≤ 15 years,496,45952,496,45952,341,155,748,0
Age gap ≤ 10 years,496,45952,496,45952,338,158,734,0


In [6]:
summary = results[["Matched Sets", "Share Listed Larger", "Median Ratio", "Permutation p-value"]].copy()
summary["95% CI"] = results.apply(lambda r: f"[{r['CI Low']:.2f}, {r['CI High']:.2f}]", axis=1)
summary = summary[["Matched Sets", "Share Listed Larger", "Median Ratio", "95% CI", "Permutation p-value"]]
display(summary.style.format({"Share Listed Larger": "{:.1%}", "Median Ratio": "{:.2f}", "Permutation p-value": "{:.4f}"}))
display(results[["Sign-test p-value", "Nonzero Gaps", "Zero Gaps"]])
summary.to_csv(TAB_DIR / "robustness_matching_summary.csv")


,Matched Sets,Share Listed Larger,Median Ratio,95% CI,Permutation p-value
Matching Specification,,,,,
Up to 3 controls,351,85.5%,7.96,"[6.07, 11.11]",0.0001
1 control,351,80.9%,9.17,"[6.72, 13.23]",0.0001
Assets ≥ USD 1,351,84.6%,8.24,"[6.50, 11.11]",0.0001
Age gap ≤ 15 years,341,84.8%,7.13,"[5.48, 10.65]",0.0001
Age gap ≤ 10 years,338,85.5%,6.87,"[5.38, 9.61]",0.0001


,Sign-test p-value,Nonzero Gaps,Zero Gaps
Matching Specification,,,
Up to 3 controls,2.396202e-44,351,0
1 control,3.168104e-33,351,0
Assets ≥ USD 1,4.365253e-42,351,0
Age gap ≤ 15 years,2.817376e-41,341,0
Age gap ≤ 10 years,7.510107e-43,338,0


 Interpretation: 
Compare the direction and magnitude of the gap, interval width, and sample retention together. Overlapping intervals do not prove equivalence. Calipers can change which listed firms receive matches, so differences need not reflect the age restriction alone. A small permutation p-value does not establish that matching removes all confounding.


- Does the direction of the gap remain the same?
- Does its magnitude change substantially?
- How many firms remain matched?
Overlapping confidence intervals alone do not prove the specifications are equivalent. Tighter age calipers can also change which listed firms enter the matched sample.

In [7]:
for name, row in results.iterrows():
    display(Markdown(
        f"**{name}:** {int(row['Matched Sets'])} matched sets; listed firms have larger assets "
        f"in {row['Share Listed Larger']:.1%} of sets. The median within-set ratio is "
        f"{row['Median Ratio']:.2f} (95% CI {row['CI Low']:.2f}–{row['CI High']:.2f}); "
        f"one-sided permutation p = {row['Permutation p-value']:.4f}."))


**Up to 3 controls:** 351 matched sets; listed firms have larger assets in 85.5% of sets. The median within-set ratio is 7.96 (95% CI 6.07–11.11); one-sided permutation p = 0.0001.

**1 control:** 351 matched sets; listed firms have larger assets in 80.9% of sets. The median within-set ratio is 9.17 (95% CI 6.72–13.23); one-sided permutation p = 0.0001.

**Assets ≥ USD 1:** 351 matched sets; listed firms have larger assets in 84.6% of sets. The median within-set ratio is 8.24 (95% CI 6.50–11.11); one-sided permutation p = 0.0001.

**Age gap ≤ 15 years:** 341 matched sets; listed firms have larger assets in 84.8% of sets. The median within-set ratio is 7.13 (95% CI 5.48–10.65); one-sided permutation p = 0.0001.

**Age gap ≤ 10 years:** 338 matched sets; listed firms have larger assets in 85.5% of sets. The median within-set ratio is 6.87 (95% CI 5.38–9.61); one-sided permutation p = 0.0001.

| Specification | Median ratio |
|---|---:|
| Up to 3 controls | 7.96 |
| 1 control | 9.17 |
| Assets ≥ USD 1 | 8.24 |
| Age gap ≤ 15 years | 7.13 |
| Age gap ≤ 10 years | 6.87 |

### State ownership: availability and proposal deviation
A reliable state-ownership or ownership-type field is required to exclude state-owned firms. Neither company names nor GUO/ISH names or IDs establish state ownership. The schema audit below lists ownership-related column names only; it does not infer ownership from their values. Until a documented indicator is supplied, the state-owned-firm exclusion is not implemented. This is a deviation from the proposal and a limitation: differences in state ownership may contribute to the observed gap. An authorised team member would need to obtain an appropriate Orbis field and verify its definition, coverage, and reference period before applying this exclusion.

In [8]:
ownership_columns = [c for c in df.columns if any(term in c.lower() for term in
                     ("state", "public author", "ownership", "owner", "guo", "ish", "government"))]
print("Ownership-related schema fields:", ownership_columns)
# Aggregate availability audit only; no firm-level records are displayed.
employment = pd.to_numeric(df["employees"], errors="coerce")
employee_audit = pd.DataFrame({"Listing status": df["listed"].map({1: "Listed", 0: "Unlisted"}),
                               "Recorded": employment.notna(), "Positive": employment.gt(0)})
display(employee_audit.groupby("Listing status")[["Recorded", "Positive"]].sum())


Ownership-related schema fields: ['ish_bvd_id', 'guo_bvd_id', 'ISH - Name', 'GUO - Name']


,Recorded,Positive
Listing status,,
Listed,267,267
Unlisted,8,8


### Employee-count limitation
Employee count is not used as an alternative outcome. The project reports availability for only eight unlisted firms; the aggregate audit above checks the current file. Such sparse coverage creates an extremely small, selected private-firm sample and cannot support a reliable employee-based robustness comparison. Report this limitation explicitly rather than presenting an employee-count analysis.
